Air temperature for all the utilities

In [ ]:
# https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_MONTHLY
# ERA5 Monthly Aggregates - Latest Climate Reanalysis Produced by ECMWF / Copernicus Climate Change Service
# in Kelvins (k)
# image collection: 'ECMWF/ERA5/MONTHLY'
# citation: Copernicus Climate Change Service (C3S) (2017): ERA5: Fifth generation of ECMWF atmospheric reanalyses of the global climate. Copernicus Climate Change Service Climate Data Store (CDS), (date of access), https://cds.climate.copernicus.eu/cdsapp#!/home
# terms of use:Please acknowledge the use of ERA5 as stated in the Copernicus C3S/CAMS License agreement:
#
# 5.1.1 Where the Licensee communicates or distributes Copernicus Products to the public, the Licensee shall inform the recipients of the source by using the following or any similar notice: "Generated using Copernicus Climate Change Service information (Year)".
# 5.1.2 Where the Licensee makes or contributes to a publication or distribution containing adapted or modified Copernicus Products, the Licensee shall provide the following or any similar notice: "Contains modified Copernicus Climate Change Service information (Year)".
# 5.1.3 Any such publication or distribution covered by clauses 5.1.1 and 5.1.2 shall state that neither the European Commission nor ECMWF is responsible for any use that may be made of the Copernicus information or Data it contains.

# band : mean_2m_air_temperature , measured in Kelvins. from 223.6 to 304 K


# lets use the bound for TEP (Tucson Electric Power) in 2014, full_id: AZ_24211


In [1]:
# import library
import ee
import os
import geemap
import geopandas as gpd
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from tqdm import tqdm


In [3]:
# authenticate & initilialize earth engine
ee.Authenticate()
ee.Initialize(project='ee-jfelix-netmetering')

In [5]:
# set working directory/mount drive
from google.colab import drive
drive.mount('/content/drive')

#  Set working directory inside Google Drive
drive_path = '/content/drive/My Drive/net_metering'
os.makedirs(drive_path, exist_ok=True)
os.chdir(drive_path)

print("Current working directory:", os.getcwd())

# to keep console from disconnecting ctr+shift+i to inspector and go to the console
# function ClickConnect(){
#    console.log("Working");
#    document.querySelector("colab-toolbar-button#connect").click()
#}
#setInterval(ClickConnect,60000)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory: /content/drive/My Drive/net_metering


In [28]:
# get the boundary for all the offices
# you have to mount the google drive
# load the shapefile for all the utility boundaries (This was simplified in R)
all_sf = gpd.read_file('shapefiles/utility_boundaries_simplified_annual_v2.shp')


/usr/local/lib/python3.11/dist-packages/pyogrio/raw.py:198: RuntimeWarning: shapefiles/utility_boundaries_simplified_annual_v2.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [15]:
all_sf['year'].max()

2023.0

In [29]:
# view the boundary shapefile
all_sf.head()
all_sf.tail()


,year,full_id,full_id_y,geometry
5148,2012.0,UT_11135,UT_11135_2012,"POLYGON ((-111.99107 41.83314, -111.64558 41.7..."
5149,2013.0,UT_11135,UT_11135_2013,"POLYGON ((-111.99107 41.83314, -111.64558 41.7..."
5150,2011.0,VA_19882,VA_19882_2011,"POLYGON ((-80.61584 37.22951, -80.35152 37.349..."
5151,2012.0,VA_19882,VA_19882_2012,"POLYGON ((-80.61584 37.22951, -80.35152 37.349..."
5152,2013.0,VA_19882,VA_19882_2013,"POLYGON ((-80.61584 37.22951, -80.35152 37.349..."


In [8]:
# bring the data from ee and select the band
temp_collection = ee.ImageCollection('ECMWF/ERA5/MONTHLY').select('mean_2m_air_temperature')

In [30]:
# List of unique utility IDs
ulist = all_sf['full_id_y'].unique().tolist()

In [62]:
# create an empty df to collect data for loop
utility_monthly_temp = []

In [63]:
# Loop over each utility
for u in tqdm(ulist):  # u = "AZ_24211_2011"
    start_time = datetime.now()

    u_sf = all_sf[all_sf['full_id_y'] == u]

    # Add an explicit check for None geometry before attempting to access its attributes
    if u_sf.empty or u_sf.geometry.is_empty.any() or u_sf.geometry.values[0] is None:
        print(f"Skipping utility {u} due to empty or invalid geometry.")
        continue

    # Convert geometry to EE
    geom_json = u_sf.geometry.values[0].__geo_interface__
    boundary = ee.Geometry(geom_json)

    # Extract year
    year = int(u_sf['year'].values[0])
    d1 = datetime(year, 1, 1)
    d2 = d1 + relativedelta(months=12)

    # Filter image collection
    monthly_image = temp_collection.filterDate(d1.strftime('%Y-%m-%d'), d2.strftime('%Y-%m-%d'))

    # Function to extract average temp for each image
    def extract_temp(image):
        date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd HH:mm')

        med_temp = image.reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=boundary,
            scale=5000,
            maxPixels=1e13
        ).get('mean_2m_air_temperature')

        mean_temp = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=boundary,
            scale=5000,
            maxPixels=1e13
        ).get('mean_2m_air_temperature')
        return ee.Feature(None, {'date': date, 'avg_temp': mean_temp, 'median_temp': med_temp})
    # Map over collection and get features
    temp_features = monthly_image.map(extract_temp)
    temp_table = ee.FeatureCollection(temp_features)

    # Export to client
    try:
        temp_dicts = geemap.ee_to_geojson(temp_table)
        temp_df = pd.DataFrame(temp_dicts['features'])
        temp_df['date'] = pd.to_datetime(temp_df['properties'].apply(lambda x: x['date']))
        temp_df['avg_temp'] = temp_df['properties'].apply(lambda x: x['avg_temp'])
        temp_df['median_temp'] = temp_df['properties'].apply(lambda x: x['median_temp'])
        temp_df.drop(columns=['properties', 'type', 'geometry'], inplace=True)

        # Add utility info
        temp_df['full_id_y'] = u
        temp_df['year'] = year
        temp_df['month'] = temp_df['date'].dt.month

        # save
        utility_monthly_temp.append(temp_df)

    except Exception as e:
        print(f"Failed for utility {u}: {e}")
        continue

    print(f"{u} completed in {datetime.now() - start_time}")

# Combine all data and write to CSV
result_df = pd.concat(utility_monthly_temp, ignore_index=True)

# Save to CSV
result_df.to_csv("data files/monthly_air_temp_utilities_kelvin.csv", index=False)

  0%|          | 1/4292 [00:00<26:37,  2.69it/s]

AK_1651_2011 completed in 0:00:00.372139


  0%|          | 2/4292 [00:00<34:02,  2.10it/s]

AK_7353_2011 completed in 0:00:00.547501


  0%|          | 3/4292 [00:01<29:24,  2.43it/s]

...

100%|█████████▉| 4291/4292 [17:28<00:00,  6.93it/s]

VA_19882_2011 completed in 0:00:00.295016
VA_19882_2012 completed in 0:00:00.182138


100%|██████████| 4292/4292 [17:28<00:00,  4.09it/s]


VA_19882_2013 completed in 0:00:00.180826


/tmp/ipython-input-2731846361.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_df = pd.concat(utility_monthly_temp, ignore_index=True)


In [86]:
# read in the dataset
result_df = pd.read_csv('data files/monthly_air_temp_utilities_kelvin.csv')

In [87]:
# check out the data
result_df.head()

,id,date,avg_temp,median_temp,full_id_y,year,month
0,201101,2011-01-01,262.901527,262.708801,AK_1651_2011,2011,1
1,201102,2011-02-01,262.651882,262.448761,AK_1651_2011,2011,2
2,201103,2011-03-01,263.460223,263.237762,AK_1651_2011,2011,3
3,201104,2011-04-01,269.190750,269.344147,AK_1651_2011,2011,4
4,201105,2011-05-01,279.746675,280.305023,AK_1651_2011,2011,5


In [88]:
# see the values that were failed or not in the u_sf
failed_utilities = [u for u in ulist if u not in result_df['full_id_y'].unique().tolist()]
print(failed_utilities)

['AK_219_2021', 'AK_3522_2021', 'AK_7353_2021', 'AK_11824_2021', 'AK_19558_2021', 'AR_814_2021', 'AR_817_2021', 'AR_3093_2021', 'AR_5860_2021', 'AR_6342_2021', 'AR_13718_2021', 'AR_14063_2021', 'AR_17698_2021', 'AZ_176_2021', 'AZ_803_2021', 'AZ_12919_2021', 'AZ_16572_2021', 'AZ_19189_2021', 'AZ_19728_2021', 'AZ_21538_2021', 'AZ_24211_2021', 'CA_4390_2021', 'CA_9216_2021', 'CA_11208_2021', 'CA_12745_2021', 'CA_14328_2021', 'CA_14354_2021', 'CA_14534_2021', 'CA_16534_2021', 'CA_16609_2021', 'CA_16655_2021', 'CA_17609_2021', 'CA_17612_2021', 'CA_18260_2021', 'CA_19281_2021', 'CO_3989_2021', 'CO_6604_2021', 'CO_9336_2021', 'CO_12866_2021', 'CO_15257_2021', 'CO_15466_2021', 'CO_16603_2021', 'CO_19499_2021', 'CO_27058_2021', 'CO_56146_2021', 'CT_4176_2021', 'CT_7716_2021', 'CT_19497_2021', 'CT_20038_2021', 'DC_15270_2021', 'DE_5027_2021', 'DE_5070_2021', 'DE_5335_2021', 'DE_13519_2021', 'FL_6452_2021', 'FL_6455_2021', 'FL_6457_2021', 'FL_7801_2021', 'FL_9617_2021', 'FL_18454_2021', 'GA_3916_

In [89]:
# find these in the all_sf
failed_df = all_sf[all_sf['full_id_y'].isin(failed_utilities)]
failed_df.head()


,year,full_id,full_id_y,geometry
3404,2021.0,AK_219,AK_219_2021,"MULTIPOLYGON (((-152.84 66.52844, -152.54921 6..."
3405,2021.0,AK_3522,AK_3522_2021,"MULTIPOLYGON (((-149.92786 61.26033, -150.1725..."
3406,2021.0,AK_7353,AK_7353_2021,"MULTIPOLYGON (((-148.99777 61.68576, -149.0274..."
3407,2021.0,AK_11824,AK_11824_2021,"MULTIPOLYGON (((-147.70998 61.25606, -144.8711..."
3408,2021.0,AK_19558,AK_19558_2021,"MULTIPOLYGON (((-150.2314 60.93248, -149.79817..."


In [92]:
# use the geometry from results that worked by the same full_id
# start with the unique full_id in failed_df
fi_ulist = failed_df['full_id'].unique().tolist()

# make the full_id in results by removing everything after first number  from full_id_y
result_df['full_id'] = result_df['full_id_y'].apply(lambda x: x.rsplit('_', 1)[0])
result_df.head()


,id,date,avg_temp,median_temp,full_id_y,year,month,full_id
0,201101,2011-01-01,262.901527,262.708801,AK_1651_2011,2011,1,AK_1651
1,201102,2011-02-01,262.651882,262.448761,AK_1651_2011,2011,2,AK_1651
2,201103,2011-03-01,263.460223,263.237762,AK_1651_2011,2011,3,AK_1651
3,201104,2011-04-01,269.190750,269.344147,AK_1651_2011,2011,4,AK_1651
4,201105,2011-05-01,279.746675,280.305023,AK_1651_2011,2011,5,AK_1651


In [93]:
# find the full_id's from fi_ulist in result_df
good_fi_df = result_df[result_df['full_id'].isin(fi_ulist)]
good_fi_df.head()

,id,date,avg_temp,median_temp,full_id_y,year,month,full_id
12,201101,2011-01-01,253.831641,254.329533,AK_7353_2011,2011,1,AK_7353
13,201102,2011-02-01,253.333961,254.224711,AK_7353_2011,2011,2,AK_7353
14,201103,2011-03-01,258.729929,259.534265,AK_7353_2011,2011,3,AK_7353
15,201104,2011-04-01,270.226517,271.338089,AK_7353_2011,2011,4,AK_7353
16,201105,2011-05-01,281.966434,282.717443,AK_7353_2011,2011,5,AK_7353


In [96]:
# use the geometries from all_sf that match what is in good_fi_df as an sf
# object
good_fi_sf = all_sf[all_sf['full_id_y'].isin(good_fi_df['full_id_y'])]
good_fi_sf

,year,full_id,full_id_y,geometry
1,2011.0,AK_7353,AK_7353_2011,"MULTIPOLYGON (((-145.49744 62.8097, -145.94969..."
2,2011.0,AK_19558,AK_19558_2011,"MULTIPOLYGON (((-151.41011 60.72023, -150.5924..."
3,2011.0,AR_814,AR_814_2011,"MULTIPOLYGON (((-89.92713 35.80674, -89.97135 ..."
4,2011.0,AR_817,AR_817_2011,"MULTIPOLYGON (((-94.4324 35.36433, -94.42681 3..."
5,2011.0,AR_6342,AR_6342_2011,"MULTIPOLYGON (((-91.90864 34.25538, -91.55306 ..."
...,...,...,...,...
5148,2012.0,UT_11135,UT_11135_2012,"POLYGON ((-111.99107 41.83314, -111.64558 41.7..."
5149,2013.0,UT_11135,UT_11135_2013,"POLYGON ((-111.99107 41.83314, -111.64558 41.7..."
5150,2011.0,VA_19882,VA_19882_2011,"POLYGON ((-80.61584 37.22951, -80.35152 37.349..."
5151,2012.0,VA_19882,VA_19882_2012,"POLYGON ((-80.61584 37.22951, -80.35152 37.349..."


In [98]:
# select the latest observation for full_id in good_fi_sf
good_fi_sf = good_fi_sf.sort_values('year', ascending=False).drop_duplicates('full_id')

# drop the year and full_id_y
good_fi_sf = good_fi_sf.drop(columns=['year', 'full_id_y'])
good_fi_sf

,full_id,geometry
3400,WY_19156,"POLYGON ((-104.24224 42.53926, -105.52642 42.2..."
5136,NH_26510,"MULTIPOLYGON (((-72.30138 43.53117, -72.2327 4..."
3384,WI_13697,"POLYGON ((-90.23122 46.08239, -90.0437 45.6686..."
3056,AR_13718,"MULTIPOLYGON (((-92.39016 34.76831, -92.42712 ..."
3055,AR_6342,"MULTIPOLYGON (((-90.84715 34.20613, -90.9498 3..."
...,...,...
3387,WI_20847,"MULTIPOLYGON (((-87.89778 43.93541, -87.79727 ..."
3386,WI_13815,"POLYGON ((-92.70405 45.35366, -92.293 46.33216..."
982,GA_7140,"MULTIPOLYGON (((-84.97732 32.29543, -84.98883 ..."
207,SD_19545,"MULTIPOLYGON (((-104.05604 44.58166, -102.9641..."


In [100]:
# replace the geometry in failed_df and attach the good geometry from good_fi_sf by full_id
failed_df['geometry'] = failed_df['full_id'].map(good_fi_sf.set_index('full_id')['geometry'])

# see the top 20 values
failed_df.head(20)


/usr/local/lib/python3.11/dist-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,year,full_id,full_id_y,geometry
3404,2021.0,AK_219,AK_219_2021,"MULTIPOLYGON (((-152.84 66.52844, -152.54921 6..."
3405,2021.0,AK_3522,AK_3522_2021,"MULTIPOLYGON (((-149.92786 61.26033, -150.1725..."
3406,2021.0,AK_7353,AK_7353_2021,"MULTIPOLYGON (((-148.99777 61.68576, -149.0274..."
3407,2021.0,AK_11824,AK_11824_2021,"MULTIPOLYGON (((-147.70998 61.25606, -144.8711..."
3408,2021.0,AK_19558,AK_19558_2021,"MULTIPOLYGON (((-150.2314 60.93248, -149.79817..."
3409,2021.0,AR_814,AR_814_2021,"MULTIPOLYGON (((-93.61086 33.01907, -94.5706 3..."
3410,2021.0,AR_817,AR_817_2021,"POLYGON ((-93.86355 34.71236, -94.5706 34.6813..."
3411,2021.0,AR_3093,AR_3093_2021,"MULTIPOLYGON (((-93.96879 36.00338, -93.96002 ..."
3412,2021.0,AR_5860,AR_5860_2021,"POLYGON ((-94.14034 36.45877, -94.56561 36.223..."
3413,2021.0,AR_6342,AR_6342_2021,"MULTIPOLYGON (((-90.84715 34.20613, -90.9498 3..."


In [102]:
# make the failed_df into an sf object
failed_df = gpd.GeoDataFrame(failed_df, geometry='geometry')
failed_df.head()


,year,full_id,full_id_y,geometry
3404,2021.0,AK_219,AK_219_2021,"MULTIPOLYGON (((-152.84 66.52844, -152.54921 6..."
3405,2021.0,AK_3522,AK_3522_2021,"MULTIPOLYGON (((-149.92786 61.26033, -150.1725..."
3406,2021.0,AK_7353,AK_7353_2021,"MULTIPOLYGON (((-148.99777 61.68576, -149.0274..."
3407,2021.0,AK_11824,AK_11824_2021,"MULTIPOLYGON (((-147.70998 61.25606, -144.8711..."
3408,2021.0,AK_19558,AK_19558_2021,"MULTIPOLYGON (((-150.2314 60.93248, -149.79817..."


In [105]:
# this requires a new image collection
#https://developers.google.com/earth-engine/datasets/catalog/NCEP_RE_surface_temp#bands

temp_collection = ee.ImageCollection("NCEP_RE/surface_temp").select('air')

In [ ]:

# create an empty df to collect data for loop
utility_monthly_temp = []

# Loop over each utility
for u in tqdm(failed_utilities):  # u = "WY_14354_2023"
    start_time = datetime.now()

    u_sf = failed_df[failed_df['full_id_y'] == u]

    # Add an explicit check for None geometry before attempting to access its attributes
    if u_sf.empty or u_sf.geometry.is_empty.any() or u_sf.geometry.values[0] is None:
        print(f"Skipping utility {u} due to empty or invalid geometry.")
        continue

    # Convert geometry to EE
    geom_json = u_sf.geometry.values[0].__geo_interface__
    boundary = ee.Geometry(geom_json)

    # Extract year
    year = int(u_sf['year'].values[0])
    d1 = datetime(year, 1, 1)
    d2 = d1 + relativedelta(months=12)

    # Filter image collection
    monthly_image = temp_collection.filterDate(d1.strftime('%Y-%m-%d'), d2.strftime('%Y-%m-%d'))

    # Function to extract average temp for each image
    def extract_temp(image):
        date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd HH:mm')

        med_temp = image.reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=boundary,
            scale=5000,
            maxPixels=1e13
        ).get('air')

        mean_temp = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=boundary,
            scale=5000,
            maxPixels=1e13
        ).get('air')
        return ee.Feature(None, {'date': date, 'avg_temp': mean_temp, 'median_temp': med_temp})
    # Map over collection and get features
    temp_features = monthly_image.map(extract_temp)
    temp_table = ee.FeatureCollection(temp_features)

    # Export to client
    try:
        temp_dicts = geemap.ee_to_geojson(temp_table)
        temp_df = pd.DataFrame(temp_dicts['features'])
        temp_df['date'] = pd.to_datetime(temp_df['properties'].apply(lambda x: x['date']))
        temp_df['avg_temp'] = temp_df['properties'].apply(lambda x: x['avg_temp'])
        temp_df['median_temp'] = temp_df['properties'].apply(lambda x: x['median_temp'])
        temp_df.drop(columns=['properties', 'type', 'geometry'], inplace=True)

        # Add utility info
        temp_df['full_id_y'] = u
        temp_df['year'] = year
        temp_df['month'] = temp_df['date'].dt.month

        # save
        utility_monthly_temp.append(temp_df)

    except Exception as e:
        print(f"Failed for utility {u}: {e}")
        continue

    print(f"{u} completed in {datetime.now() - start_time}")

# Combine all data and write to CSV
missing_df = pd.concat(utility_monthly_temp, ignore_index=True)

# Save to CSV
missing_df.to_csv("data files/missing_monthly_air_temp_utilities_kelvin.csv", index=False)



  0%|          | 1/1508 [00:03<1:23:40,  3.33s/it]

AK_219_2021 completed in 0:00:03.330988


  0%|          | 2/1508 [00:05<1:01:22,  2.44s/it]

AK_3522_2021 completed in 0:00:01.821312


  0%|          | 3/1508 [00:12<1:59:34,  4.77s/it]

...

NH_26510_2021 completed in 0:00:09.459221


100%|█████████▉| 1503/1508 [1:53:49<00:38,  7.61s/it]

NH_26510_2022 completed in 0:00:08.647707


100%|██████████| 1508/1508 [1:53:57<00:00,  4.53s/it]

NH_26510_2023 completed in 0:00:08.148567
Skipping utility TN_3704_2012 due to empty or invalid geometry.
Skipping utility TN_3704_2013 due to empty or invalid geometry.
Skipping utility TN_3758_2012 due to empty or invalid geometry.
Skipping utility TN_3758_2013 due to empty or invalid geometry.


Download URL: https://earthengine.googleapis.com/v1/projects/ee-jfelix-netmetering/tables/f40b05a0d96e6057fab1acba04670ab7-511aaa63de20224b7ede2284d8f4565a:getFeatures
